In [2]:
import cloudscraper
from bs4 import BeautifulSoup
import csv
import time

BASE_URL = "https://ai.stackexchange.com/questions?tab=newest&page="

scraper = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)

all_data = []

page = 1
max_pages = 300

while page <= max_pages:
    print(f"\n===============================")
    print(f" ĐANG SCRAPE PAGE {page} ...")
    print(f"===============================")

    url = BASE_URL + str(page)
    response = scraper.get(url)

    print("Status:", response.status_code)

    if response.status_code != 200:
        print(" Bị chặn hoặc page không tồn tại → dừng tại page:", page)
        break
    
    soup = BeautifulSoup(response.text, "html.parser")
    questions = soup.select(".s-post-summary")

    if not questions:
        print(" Không còn câu hỏi → dừng tại page:", page)
        break

    print(" Số câu hỏi tìm thấy:", len(questions))

    for q in questions:

        title_tag = q.select_one(".s-link")
        if not title_tag:
            continue

        title = title_tag.get_text(strip=True)
        link = "https://ai.stackexchange.com" + title_tag["href"]

        # Lấy toàn bộ stats (votes – answers – views)
        stat_items = q.select(".s-post-summary--stats-item-number")

        votes  = stat_items[0].get_text(strip=True) if len(stat_items) > 0 else "0"
        answers = stat_items[1].get_text(strip=True) if len(stat_items) > 1 else "0"
        views  = stat_items[2].get_text(strip=True) if len(stat_items) > 2 else "0"

        tags = [t.get_text(strip=True) for t in q.select(".post-tag")]

        all_data.append({
            "page": page,
            "title": title,
            "link": link,
            "votes": votes,
            "answers": answers,
            "views": views,
            "tags": ", ".join(tags)
        })

    time.sleep(1.2)
    page += 1

# ================================
# LƯU CSV
# ================================
csv_file = "questions_with_stats.csv"

with open(csv_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["page", "title", "link", "votes", "answers", "views", "tags"]
    )
    writer.writeheader()
    writer.writerows(all_data)

print("\n DONE! Tổng số câu hỏi scrape được:", len(all_data))
print(f" File CSV đã tạo: {csv_file}")



 ĐANG SCRAPE PAGE 1 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 2 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 3 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 4 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 5 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 6 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 7 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 8 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 9 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 10 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 11 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 12 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 13 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 14 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 15 ...
Status: 200
 Số câu hỏi tìm thấy: 15

 ĐANG SCRAPE PAGE 16 ...
Status: 200
 Số câu hỏi tìm thấy: 15



In [3]:
import pandas as pd
df = pd.read_csv("questions_with_stats.csv")
df.head()

,page,title,link,votes,answers,views,tags
0,1,"If LLMs like OpenAI / DeepSeek / Gemini exist,...",https://ai.stackexchange.com/questions/49110/i...,0,2,24,"natural-language-processing, pytorch, large-la..."
1,1,Most efficient way to put together different i...,https://ai.stackexchange.com/questions/49109/m...,0,1,19,"neural-networks, training, input-layer"
2,1,Why do AI language models overuse em dashes co...,https://ai.stackexchange.com/questions/49107/w...,1,1,33,"large-language-models, training-datasets, text..."
3,1,Beyond DPI: Can Reference-Free LLM Judge Certi...,https://ai.stackexchange.com/questions/49106/b...,0,0,13,"machine-learning, information-theory, mechanis..."
4,1,Open AI Electricity Requirement Matrix Multipl...,https://ai.stackexchange.com/questions/49104/o...,0,1,36,"open-ai, gpu, linear-algebra, hardware"
